# 가중 교차엔트로피 U-Net — 원문제 모범답안2 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 **Individual Radar** *모범답안2*(최고점 0.7963→**0.9834**) 와 동일): **다중클래스 U-Net** 을
**가중 교차엔트로피**(`CrossEntropyLoss(weight=…)`)로 학습한다. **이 노트북 코드는 Radar 모범답안2 골격을 그대로** 따른다:

| Radar 모범답안2 | 이 노트북 |
|---|---|
| `class RadarDS(Dataset)` → (img, label) | `class SegDS(Dataset)` (동일 구조) |
| `class DC` / `class UNet`(cout=5) | 동일 (cout=3) |
| **`crit = nn.CrossEntropyLoss(weight=tensor([1,50,50,50,50]))`** | **`weight=tensor([1,5,5])`** (배경 1·비배경 강조) |
| `DataLoader` + train 루프 + argmax 예측 | 동일 |

**데이터**: [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018). 이진 마스크를 **3클래스**(배경/
핵코어/경계)로 바꿔 다중클래스 가중 CE 를 연습한다(핵코어가 ~3% 로 희소).

> ⚠️ **정직한 한계**: Radar 는 클래스가 *극도로 희소 + 채점이 비배경 가중* 이라 가중 CE 가 0.79→0.98 로 크게 도움된다.
> DSB 는 불균형이 약하고 지표가 *균등 mIoU* 라 가중 효과가 작다(무가중과 비슷). **여기선 모범답안의 *코드 패턴*
> 을 익히는 게 목적** — 가중이 유리한 건 *채점/목표가 희소 클래스를 중시할 때*다.

> ⚙️ GPU 권장(작은 U-Net, T4 ~3분).  ⚠️ API 토큰 평문 — 외부 공유 금지. 09(이진 BCE+Dice)와 손실/클래스 처리 대비.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "scipy", "pillow", "scikit-image"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 데이터 다운로드 (stage1_train / stage2_test_final)

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
for fn in ["stage1_train.zip", "stage2_test_final.zip"]:
    api.competition_download_file("data-science-bowl-2018", fn, path=WORK_DIR, quiet=False)
for fn, out in [("stage1_train.zip", "stage1_train"), ("stage2_test_final.zip", "stage2_test_final")]:
    zp = os.path.join(WORK_DIR, fn)
    with zipfile.ZipFile(zp) as zf: zf.extractall(os.path.join(WORK_DIR, out))
    os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "stage1_train")
TEST_DIR  = os.path.join(WORK_DIR, "stage2_test_final")
print("train:", len(os.listdir(TRAIN_DIR)), "| test:", len(os.listdir(TEST_DIR)))

## 2. 3클래스 라벨맵 (배경 0 / 핵코어 1 / 경계 2)
각 핵 마스크를 침식해 코어(희소)와 경계를 나눈다 → Radar 의 다중클래스 라벨처럼 만든다.

In [ ]:
from PIL import Image
from scipy.ndimage import binary_erosion
SZ = 128; ERODE = 2; ST = np.ones((3, 3), bool)

def load_sample(d):
    ip = glob.glob(os.path.join(d, "images", "*.png"))[0]
    img = np.asarray(Image.open(ip).convert("RGB").resize((SZ, SZ)), "float32").transpose(2, 0, 1) / 255.
    lab = np.zeros((SZ, SZ), "int64")
    for mp in glob.glob(os.path.join(d, "masks", "*.png")):
        m = np.asarray(Image.open(mp).convert("L").resize((SZ, SZ), Image.NEAREST)) > 0
        if m.sum() == 0: continue
        lab[m] = 2; lab[binary_erosion(m, ST, iterations=ERODE)] = 1
    return img, lab

ids = sorted(os.listdir(TRAIN_DIR))
X = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[0] for i in ids]).astype("float32")
Y = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[1] for i in ids])
print("X:", X.shape, "| 클래스 비율(배경/코어/경계):", np.round(np.bincount(Y.reshape(-1), minlength=3)/Y.size, 3))

## 3. Dataset & DataLoader (Radar 모범답안2 `RadarDS` 구조 그대로)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class SegDS(Dataset):
    def __init__(self, X, Y, lbl=True): self.X = X; self.Y = Y; self.lbl = lbl
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.tensor(self.X[i])
        if self.lbl: return x, torch.tensor(self.Y[i], dtype=torch.long)
        return x

tr_i, va_i = train_test_split(np.arange(len(X)), test_size=0.1, random_state=42)
train_dl = DataLoader(SegDS(X[tr_i], Y[tr_i]), batch_size=16, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("train:", len(tr_i), "| valid:", len(va_i))

## 4. DC / UNet (Radar 모범답안2 그대로, cout=3)

In [ ]:
import torch.nn as nn

class DC(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.n = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                               nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))
    def forward(self, x): return self.n(x)

class UNet(nn.Module):
    def __init__(self, cin=3, cout=3, b=32):
        super().__init__()
        self.e1 = DC(cin, b); self.e2 = DC(b, b*2); self.e3 = DC(b*2, b*4); self.bo = DC(b*4, b*8); self.pool = nn.MaxPool2d(2)
        self.u3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2); self.d3 = DC(b*8, b*4)
        self.u2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2); self.d2 = DC(b*4, b*2)
        self.u1 = nn.ConvTranspose2d(b*2, b, 2, stride=2);   self.d1 = DC(b*2, b); self.o = nn.Conv2d(b, cout, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2)); bo = self.bo(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(bo), e3], 1)); d2 = self.d2(torch.cat([self.u2(d3), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.o(d1)

## 5. 학습 — 가중 CrossEntropyLoss (모범답안2의 핵심 한 줄)
`crit = nn.CrossEntropyLoss(weight=…)` — 배경 1, 희소 비배경 클래스를 강조(모범답안은 `[1,50,50,50,50]`).

In [ ]:
def iou_per_class(p, g, n=3):
    return [float(((p==k)&(g==k)).sum()) / float(((p==k)|(g==k)).sum() or 1) for k in range(n)]

model = UNet().to(device)
crit = nn.CrossEntropyLoss(weight=torch.tensor([1., 5., 5.], device=device))   # 배경 1, 비배경 강조
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 15
for ep in range(EPOCHS):
    model.train(); tot = 0
    for img, lbl in train_dl:
        img, lbl = img.to(device), lbl.to(device)
        opt.zero_grad(); loss = crit(model(img), lbl); loss.backward(); opt.step(); tot += loss.item()
    if (ep+1) % 5 == 0: print(f"epoch {ep+1}/{EPOCHS}  loss {tot/len(train_dl):.4f}")

# 검증 per-class IoU
model.eval(); P, G = [], []
with torch.no_grad():
    for i in range(0, len(va_i), 16):
        b = va_i[i:i+16]
        P.append(model(torch.tensor(X[b]).to(device)).argmax(1).cpu().numpy()); G.append(Y[b])
io = iou_per_class(np.concatenate(P), np.concatenate(G))
print(f"val IoU  배경 {io[0]:.3f}  코어 {io[1]:.3f}  경계 {io[2]:.3f}  | mIoU {np.mean(io):.3f}")
print("→ 다중클래스 U-Net + 가중 CrossEntropyLoss — Radar 모범답안2 의 핵심 기법(코드) 그대로.")

## 6. 캐글 제출 — 핵(class≥1) 연결요소 → 인스턴스 RLE
Radar 제출은 픽셀별 클래스 CSV 지만, DSB 는 **인스턴스 RLE** 라 핵 영역의 연결요소를 라벨링한다(09 와 동일 방식).

In [ ]:
import pandas as pd
from skimage.morphology import label

def rle_encode(mask):
    px = mask.flatten(order="F"); px = np.concatenate([[0], px, [0]])
    runs = np.where(px[1:] != px[:-1])[0] + 1; runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

test_ids = sorted(os.listdir(TEST_DIR)); rows = []
model.eval()
for sid in test_ids:
    paths = glob.glob(os.path.join(TEST_DIR, sid, "images", "*.png")); found = False
    if paths:
        orig = Image.open(paths[0]).convert("RGB"); Wd, Ht = orig.size
        x = np.asarray(orig.resize((SZ, SZ)), "float32").transpose(2, 0, 1) / 255.
        with torch.no_grad():
            pr = model(torch.tensor(x[None]).to(device)).argmax(1)[0].cpu().numpy()
        fg = np.asarray(Image.fromarray(((pr >= 1).astype("uint8") * 255)).resize((Wd, Ht))) > 127
        lab = label(fg)
        for k in range(1, lab.max() + 1):
            rle = rle_encode(lab == k)
            if rle: rows.append({"ImageId": sid, "EncodedPixels": rle}); found = True
    if not found: rows.append({"ImageId": sid, "EncodedPixels": "1 1"})

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"]).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(rows), "| unique ids:", pd.DataFrame(rows)["ImageId"].nunique(), "/", len(test_ids))

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018/submit) 에 제출.

Radar **모범답안2**(최고점) 골격(`SegDS`→`DC`/`UNet`→**`CrossEntropyLoss(weight=…)`**→train→argmax 예측)을 그대로
옮겼다. 핵심은 *다중클래스 U-Net 을 가중 CE 로 학습* — 단, **가중 효과는 채점/목표가 희소 클래스를 중시할 때 큼**
(Radar 0.79→0.98). DSB 의 균등 mIoU 엔 효과가 작으니 *코드 패턴* 위주로 보면 된다. 더: 가중치 값 조정, CE+Dice,
코어를 watershed 마커로 인스턴스 분리, 해상도↑.